# 00_setup: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/00_setup/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Tour the catalog and inspect literal phrase matching.

Concepts: IDs, rows, text search.
Run: uv run python modules/00_setup/examples/01_from_scratch.py
Expected: 1,000 rows; phrase matches: 0
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

from common.data import load_products, search_queries


def literal_ids(rows: list[dict], text: str) -> list[str]:
    # KEY IDEA: characters must match; this procedure has no representation of intent.
    return [row["id"] for row in rows if text.casefold() in row["description"].casefold()]


def main() -> None:
    products = load_products(limit=1000)
    query = search_queries()[0]["text"]
    print(products[["id", "title", "price", "region"]].head().to_string(index=False))
    print(
        f"Rows: {len(products):,}; phrase matches: {len(literal_ids(products.to_dict('records'), query))}"
    )
    print("The input is a need. A phrase matcher cannot infer a heated coaster.")


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/00_setup/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Give Chroma explicit precomputed vectors.

    Concepts: embedded database, IDs, embedding function boundary.
    Run: uv run --extra embedded python modules/00_setup/examples/02_production.py
    Expected: five ceramic mug-warmer results
    CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
    Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
    """

    # course-extra: embedded
    import time
    import uuid

    import chromadb

    from common.pretty import show_hits
    from common.teaching import catalog, ranked_hits

    def main() -> None:
        products, vectors, query, _ = catalog()
        client = chromadb.EphemeralClient(settings=chromadb.Settings(anonymized_telemetry=False))
        name = "marlowe_" + uuid.uuid4().hex
        collection = client.create_collection(
            name, embedding_function=None, configuration={"hnsw": {"space": "cosine"}}
        )
        try:
            # KEY IDEA: the same vectors enter both engines; Chroma does not choose a model for us.
            collection.add(ids=products["id"].tolist(), embeddings=vectors.tolist())
            start = time.perf_counter()
            result = collection.query(query_embeddings=[query.tolist()], n_results=5)
            elapsed = (time.perf_counter() - start) * 1000
            ranked = list(
                zip(result["ids"][0], [1 - d for d in result["distances"][0]], strict=True)
            )
            show_hits(ranked_hits(products, ranked), elapsed, title="Chroma explicit-vector search")
        finally:
            client.delete_collection(name)

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/00_setup/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure the motivating exact-search session.

Concepts: phrase baseline, useful hits, repeated timing.
Run: uv run python modules/00_setup/examples/03_measure.py
Expected: 0 literal matches; 5 mug warmers; evidence JSON
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

from common.apps.search import open_catalog, search_saved
from common.pretty import show_hits
from common.teaching import evidence, timed


def main() -> None:
    store, products, _ = open_catalog(limit=1000)
    try:
        query, hits, elapsed = search_saved(store)
        literal = int(
            products["description"].str.contains(query["text"], case=False, regex=False).sum()
        )
        useful = sum("mug warmer" in hit.metadata["title"] for hit in hits)
        # KEY IDEA: a successful motivating query is evidence for this query, not a whole benchmark.
        assert literal == 0 and useful == 5
        show_hits(hits, elapsed, title="First useful search")
        evidence(
            "orientation",
            {
                "literal_matches": literal,
                "mug_warmers_at_5": useful,
                "saved_search_including_query_load": timed(lambda: search_saved(store)),
            },
        )
    finally:
        store.close()


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import catalog

products, _, _, _ = catalog(100)
products[["id", "title", "price"]].head()

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.